In [2]:
import os

base_dir = '/kaggle/input/animal-faces/afhq/train/cat'
all_image_urls = os.listdir(base_dir)
# all_image_urls[:10]

In [3]:
sample_image_urls = all_image_urls[:250]
sample_image_urls = list(map(lambda item: f'{base_dir}/{item}', sample_image_urls))
# sample_image_urls

In [4]:
import pandas as pd
from PIL import Image

payloads = pd.DataFrame.from_records({"image_url":sample_image_urls})
payloads['type']='cat'
payloads


,image_url,type
0,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
1,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
2,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
3,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
4,/kaggle/input/animal-faces/afhq/train/cat/flic...,cat
...,...,...
245,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
246,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
247,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat
248,/kaggle/input/animal-faces/afhq/train/cat/flic...,cat


In [5]:
images = list(map(lambda el: Image.open(el), payloads['image_url']))
# images

In [6]:
# Since the images will be in vector in DB. well create base64 string and store it along
# with metadata. helpful for previewing in the app
from io import BytesIO
import math
import base64

target_width = 256

def resize_image(image_url):
    pil_image = Image.open(image_url)
    aspect_ratio = pil_image.width / pil_image.height
    resized_pil_image = pil_image.resize([target_width, math.floor(target_width*aspect_ratio)])

    return resized_pil_image

def convert_img_to_base64(image):
    image_data = BytesIO()
    image.save(image_data, format='JPEG')
    base64_str = base64.b64encode(image_data.getvalue()).decode('utf-8')  # Decode base64 result to a UTF-8 string
    return base64_str

In [7]:
resized_images = list(map(lambda el: resize_image(el), sample_image_urls))
base64_strings = list(map(lambda el: convert_img_to_base64(el), resized_images))
payloads['base64'] = base64_strings
payloads

,image_url,type,base64
0,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
1,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
2,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
3,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
4,/kaggle/input/animal-faces/afhq/train/cat/flic...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
...,...,...,...
245,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
246,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
247,/kaggle/input/animal-faces/afhq/train/cat/pixa...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
248,/kaggle/input/animal-faces/afhq/train/cat/flic...,cat,/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...


In [8]:
from transformers import AutoImageProcessor, ResNetForImageClassification

processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")
model = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")

inputs = processor(list(images), return_tensors='pt')

outputs = model(**inputs)
embeddings = outputs.logits
embeddings

preprocessor_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.6k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

tensor([[ -8.8185,  -8.5249,  -8.4554,  ...,  -9.0750,  -6.8861,  -7.2990],
        [ -8.7549, -10.1635, -10.1358,  ...,  -9.5130,  -7.8688,  -9.1120],
        [-10.7088, -11.2569, -11.3992,  ..., -11.0634, -10.4576,  -8.7202],
        ...,
        [ -9.7486, -10.6051, -11.1507,  ...,  -9.8723,  -8.2253,  -9.3420],
        [ -9.6666,  -9.9563, -10.7314,  ...,  -9.8563,  -5.6230,  -7.1104],
        [-10.1637, -11.9126, -11.0427,  ...,  -9.9719,  -8.7289,  -9.0441]],
       grad_fn=<AddmmBackward0>)

In [9]:
embedding_len = len(embeddings[0])
embedding_len

1000

In [ ]:
QDRANT_DB_URL='https://0cef25e0-9d34-416f-a362-f8ed3c6e61a9.us-west-1-0.aws.cloud.qdrant.io'
QDRANT_API_KEY='dHT9IqabKhBnzWOtQCa5jFfOWQj5H1Y3TG8c68ov8wW5sLL5f33lMw'

In [32]:
!pip -q install qdrant_client

In [17]:
from qdrant_client import QdrantClient
import os

qclient = QdrantClient(url=QDRANT_DB_URL, api_key=QDRANT_API_KEY)
qclient

In [20]:
from qdrant_client.models import VectorParams, Distance

collection_name = 'animal_images'
collection = qclient.recreate_collection(collection_name=collection_name, vectors_config=VectorParams(size=embedding_len, distance=Distance.COSINE))

collection

<ipython-input-20-663864389543>:4: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  collection = qclient.recreate_collection(collection_name=collection_name, vectors_config=VectorParams(size=embedding_len, distance=Distance.COSINE))


True

In [23]:
payload_dicts = payloads.to_dict(orient='records')
# payload_dicts[:2]

In [29]:
from qdrant_client import models

records = [models.Record(id=idx, payload=payload_dicts[idx], vector=embeddings[idx]) for idx, _ in enumerate(payload_dicts)]

In [31]:
qclient.upload_records(collection_name=collection_name, records=records)

<ipython-input-31-4daa0f1ce5e1>:1: DeprecationWarning: `upload_records` is deprecated, use `upload_points` instead
  qclient.upload_records(collection_name=collection_name, records=records)
